# Figure 2 — Data footprint & tiling

Euclid VIS mosaic backdrop with the JAISP tiling overlaid, colored by tract patch.
ECDFS (in-distribution, tract 5063) and EDF-S (first OOD sample, tract 2394), each at
its own footprint with a shared 5' scale bar.

Saves to `paper/figures/fig2_coverage.png`. Self-contained: downsamples the VIS tiles into a
cached mosaic (`paper_figures/_fig2_cache/`) on first run, then reuses it.

In [ ]:
import numpy as np, glob, re, os
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ImageNormalize, AsinhStretch, PercentileInterval

def find_repo_root():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / 'data').is_dir() and (c / 'paper').is_dir():
            return c
    raise RuntimeError('repo root not found')

REPO = find_repo_root()
FIGDIR = REPO / 'paper' / 'figures'; FIGDIR.mkdir(exist_ok=True)
CACHE = REPO / 'paper_figures' / '_fig2_cache'; CACHE.mkdir(exist_ok=True)
FIELDS = {'ecdfs': REPO / 'data' / 'euclid_tiles_all',
          'edfs':  REPO / 'data' / 'edf_s_ood' / 'euclid_tiles_edfs'}
plt.rcParams.update({'font.size': 9, 'axes.linewidth': 0.8})
print('repo:', REPO)

In [ ]:
# --- build (or load) a downsampled VIS mosaic + per-tile sky extent + patch id ---
PAT = re.compile(r'tile_x(?P<x>\d+)_y(?P<y>\d+)(?:_tract(?P<tract>[^_]+)_patch_(?P<patch>[^_.]+))?')
BLK = 8   # block-mean downsample (1084 -> ~135 px/tile)

def block_mean(a, k):
    ny, nx = a.shape; a = a[:ny // k * k, :nx // k * k]
    return a.reshape(a.shape[0] // k, k, a.shape[1] // k, k).mean(axis=(1, 3))

def build_thumbs(tag, directory):
    cache = CACHE / f'vis_thumbs_{tag}.npz'
    if cache.exists():
        z = np.load(cache, allow_pickle=True)
        return z['thumbs'], z['extents'], z['patches']
    files = sorted(glob.glob(str(directory / 'tile_x*_euclid.npz')))
    thumbs, extents, patches = [], [], []
    for i, f in enumerate(files):
        d = np.load(f, allow_pickle=True)
        v = np.asarray(d['img_VIS'], np.float32); ny, nx = v.shape
        w = WCS(fits.Header.fromstring(d['wcs_VIS'].item()))
        ra_l, _ = w.all_pix2world(0, ny / 2, 0); ra_r, _ = w.all_pix2world(nx, ny / 2, 0)
        _, dec_b = w.all_pix2world(nx / 2, 0, 0); _, dec_t = w.all_pix2world(nx / 2, ny, 0)
        thumbs.append(block_mean(v, BLK).astype(np.float32))
        extents.append([float(ra_l), float(ra_r), float(dec_b), float(dec_t)])
        m = PAT.search(os.path.basename(f)); patches.append(m.group('patch') if (m and m.group('patch')) else 'edfs')
        if i % 200 == 0: print(f'{tag}: {i}/{len(files)}', flush=True)
    thumbs, extents, patches = np.array(thumbs), np.array(extents), np.array(patches)
    np.savez_compressed(cache, thumbs=thumbs, extents=extents, patches=patches)
    return thumbs, extents, patches

def union_area_deg2(extents, n=1500):
    ra = np.sort(extents[:, :2], 1); dec = np.sort(extents[:, 2:], 1)
    rg = np.linspace(ra.min(), ra.max(), n); dg = np.linspace(dec.min(), dec.max(), n)
    cov = np.zeros((n, n), bool)
    for (r0, r1), (d0, d1) in zip(ra, dec):
        ri = np.searchsorted(rg, [r0, r1]); di = np.searchsorted(dg, [d0, d1]); cov[di[0]:di[1], ri[0]:ri[1]] = True
    return float((cov * (rg[-1]-rg[0])/n * (dg[-1]-dg[0])/n * np.cos(np.deg2rad(dg))[:, None]).sum())

data = {tag: build_thumbs(tag, d) for tag, d in FIELDS.items()}
for tag, (th, ex, pa) in data.items():
    print(f'{tag}: {len(th)} tiles, {len(set(pa.tolist()))} patches, union={union_area_deg2(ex):.3f} deg^2')

In [ ]:
# ---- config you can tweak ----
FILL_ALPHA = 0.06          # per-patch color tint over the imagery
EDGE_ALPHA, EDGE_LW = 0.40, 0.25
STRETCH, PCT = 0.06, 99.3  # asinh stretch softening; percentile clip for the VIS backdrop
SCALEBAR_ARCMIN = 5.0

def field_center(ex):
    return 0.5 * (ex[:, :2].max() + ex[:, :2].min()), 0.5 * (ex[:, 2:].min() + ex[:, 2:].max())

def draw(ax, tag, color_by_patch, label):
    thumbs, extents, patches = data[tag]
    norm = ImageNormalize(np.concatenate([t.ravel() for t in thumbs[::20]]),
                          interval=PercentileInterval(PCT), stretch=AsinhStretch(STRETCH))
    for t, e in zip(thumbs, extents):
        ax.imshow(t, extent=[e[0], e[1], e[2], e[3]], origin='lower', cmap='gray',
                  norm=norm, interpolation='nearest', zorder=1)
    cmap = plt.cm.tab10; cols = {p: cmap(i % 10) for i, p in enumerate(sorted(set(patches.tolist())))}
    fc = [cols[p] if color_by_patch else cmap(1) for p in patches]
    rects = [Rectangle((max(e[0], e[1]), e[2]), -abs(e[0]-e[1]), abs(e[3]-e[2])) for e in extents]
    ax.add_collection(PatchCollection(rects, facecolor=fc, edgecolor='none', alpha=FILL_ALPHA, zorder=2))
    ax.add_collection(PatchCollection([Rectangle(r.get_xy(), r.get_width(), r.get_height()) for r in rects],
                                      facecolor='none', edgecolor=fc, lw=EDGE_LW, alpha=EDGE_ALPHA, zorder=3))
    rc, dc = field_center(extents)
    ra0, ra1 = extents[:, :2].max(), extents[:, :2].min(); dec0, dec1 = extents[:, 2:].min(), extents[:, 2:].max()
    ax.set_xlim(ra0 + 0.03*(ra0-ra1), ra1 - 0.03*(ra0-ra1)); ax.set_ylim(dec0 - 0.03*(dec1-dec0), dec1 + 0.03*(dec1-dec0))
    ax.set_aspect(1/np.cos(np.deg2rad(dc))); ax.set_anchor('N')
    ax.set_xlabel('R.A. [deg]'); ax.set_ylabel('Dec. [deg]'); ax.tick_params(labelsize=8)
    (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
    L = (SCALEBAR_ARCMIN/60.0)/np.cos(np.deg2rad(dc)); xb = x1 + 0.06*(x0-x1); xa = xb + L; yb = y0 + 0.06*(y1-y0)
    ax.plot([xa, xb], [yb, yb], color='w', lw=2.6, solid_capstyle='butt', zorder=5)
    ax.text((xa+xb)/2, yb + 0.02*(y1-y0), f"{SCALEBAR_ARCMIN:.0f}'", ha='center', va='bottom', fontsize=7.5, color='w', zorder=5)
    ax.text(0.03, 0.97, label, transform=ax.transAxes, fontsize=8, va='top', ha='left', color='w', zorder=6,
            bbox=dict(boxstyle='round,pad=0.3', fc='black', ec='none', alpha=0.55))
    return cols

fig, (axb, axc) = plt.subplots(1, 2, figsize=(7.1, 3.9))
fig.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.12, wspace=0.27)
draw(axb, 'ecdfs', True,  f'{len(data["ecdfs"][0])} tiles · 0.17 deg$^2$')
draw(axc, 'edfs',  False, f'{len(data["edfs"][0])} tiles · 0.018 deg$^2$')
axb.set_title('ECDFS  ·  tract 5063  ·  in-distribution', fontsize=9)
axc.set_title('EDF-S  ·  tract 2394  ·  first OOD sample', fontsize=9)
axb.text(0.97, 0.03, 'colored by tract patch', transform=axb.transAxes, fontsize=6.8, va='bottom', ha='right', color='w', alpha=0.85)

out = FIGDIR / 'fig2_coverage.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
print('saved', out)
plt.show()